## ⚠️ Note for the revision3 (BIOINF-2025-2011.R2) response

This notebook mixes two different evaluation paths: a helper function that
builds `edge_index_val` from `torch.arange(N).repeat(2, 1)` (self-loops
only -- the bug Reviewer 2 flagged, where the GNN's message passing reduces
to a per-cell MLP at eval), and a separate `NeighborLoader`-based path
(`full_loader = NeighborLoader(...)`) that does use real graph neighbors.
Any metric or embedding produced via the self-loop helper in this notebook
should not be treated as validated -- only outputs from the `NeighborLoader`
path, or from `train_v2.py`/`train_v2b.py`, reflect what the model actually
does with the graph.

This notebook was not patched in place for the revision3 response (same
reasoning as `scMeta_5foldCV.ipynb`/`scMeta_LOOCV.ipynb`: fixing it here
would duplicate the already-correct implementation elsewhere). For the
downstream biology reported in the response letter, use:

- `../run_gsea.py` (all three scenarios) for gradient/saliency-based feature
  importance, via real inductive inference (`inductive_saliency.py`).
- `../run_sub_benchmark.py` / `../gsea_sub_benchmark_v2_summary/Master_EMT_Benchmark_Summary.csv`
  for the per-cluster, per-method EMT benchmark (Wilcoxon DEG vs baseline LR
  vs scMeta gradients).
- `../draw_umap.py` for the UMAP highlighting the EMT-enriched cluster
  (cluster 56, not the old cluster 59 -- see `../run_gsea_local_subpop_v2.log`).


In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as skl
import anndata as ann
import random, os
from scipy.stats import pearsonr as pr
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score as f1
from sklearn.metrics import precision_recall_curve as prc
from sklearn.metrics import silhouette_score as sil
from sklearn.metrics import auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, average_precision_score
from sklearn.metrics import silhouette_score
from torch_geometric.nn import TransformerConv
from torch_geometric.data import Data
import gc, psutil

In [ ]:
import torch
import torch_geometric
from torch_geometric.nn import GraphConv, GCNConv
from sklearn.model_selection import StratifiedKFold
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import torch_geometric.nn as pyg_nn
from torch_geometric.data import Data
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.preprocessing import LabelEncoder
from torch_geometric.loader import NeighborLoader


In [ ]:
sc.set_figure_params(dpi=200)

In [ ]:
import torch
from torch_geometric.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
from tqdm import tqdm  # For progress bar
from sklearn.model_selection import KFold

# Set random seeds for reproducibility
import random
import numpy as np
import torch

def set_random_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)  # if using CUDA
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_random_seeds(42)
# Check if GPU is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
from torch.nn import Linear, Dropout
from torch_geometric.nn import TransformerConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score, accuracy_score


In [ ]:
from torch_geometric.data import Data
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

In [ ]:
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from torch_geometric.loader import RandomNodeLoader
from torch_geometric.utils import subgraph

In [ ]:
def memory_usgae():
    gc.collect()
    torch.cuda.empty_cache()
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024**3  # in GB

    print(f"Current memory usage: {memory_gb:.2f} GB")

# Load data

In [ ]:
# load the data
ad = sc.read_h5ad('../../Data/Cancer_cell_data_reprocessed/All_integrated.hallmark.harmony.h5ad')
ad

Label unification:

Primary: Non-metastatic Local

Local: Metastatic Local

Distant: Metastatic distant

In [ ]:
# Map cancer type → its "primary tissue"
primary_tissue_map = {
    "Breast Cancer": "Breast",
    "Lung Cancer": "Lung",
    "Ovarian Cancer": "Ovary",
    "Colorectal Cancer": "Colon"
}

# Initialize as primary
ad.obs["source"] = "Primary"


# Loop through rows
for idx, row in ad.obs.iterrows():
    cancer_type = row["Final_cancer_type"]
    tissue = row["Final_tissue"]

    primary_site = primary_tissue_map.get(cancer_type, None)

    if row["Primary_or_Metastatic"] == "Primary":
        ad.obs.at[idx, "source"] = "Primary"

    else:  # Metastatic
        if tissue == primary_site:
            ad.obs.at[idx, "source"] = "local"
        else:
            ad.obs.at[idx, "source"] = "distant"

# ✅ Summary:
print(ad.obs["source"].value_counts())


In [ ]:
memory_usgae()

# Model

In [ ]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Dropout
from torch_geometric.nn import TransformerConv, GATConv, SAGEConv


class scMeta(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, conv_type='TransformerConv', heads=4, dropout=0.3):
        """
        Flexible scMeta model with multiple GNN layer options.
        
        Parameters:
        -----------
        input_dim : int
            Input feature dimension
        hidden_dim : int
            Hidden layer dimension
        num_classes : int
            Number of output classes
        conv_type : str
            Type of graph convolution layer. Options: 'TransformerConv', 'GATConv', 'SAGEConv'
            Default: 'TransformerConv'
        heads : int
            Number of attention heads (for TransformerConv and GATConv only)
            Default: 4
        dropout : float
            Dropout rate
            Default: 0.3
        """
        super(scMeta, self).__init__()
        
        self.conv_type = conv_type
        
        # Select GNN layer based on conv_type
        if conv_type == 'TransformerConv':
            self.conv1 = TransformerConv(
                in_channels=input_dim, 
                out_channels=hidden_dim, 
                heads=heads, 
                dropout=dropout
            )
            self.conv2 = TransformerConv(
                in_channels=hidden_dim * heads, 
                out_channels=hidden_dim, 
                heads=1, 
                dropout=dropout
            )
            
        elif conv_type == 'GATConv':
            self.conv1 = GATConv(
                in_channels=input_dim, 
                out_channels=hidden_dim, 
                heads=heads, 
                dropout=dropout
            )
            self.conv2 = GATConv(
                in_channels=hidden_dim * heads, 
                out_channels=hidden_dim, 
                heads=1, 
                dropout=dropout
            )
            
        elif conv_type == 'SAGEConv':
            # SAGEConv doesn't use heads parameter
            self.conv1 = SAGEConv(
                in_channels=input_dim, 
                out_channels=hidden_dim * heads  # Scale output to match other models
            )
            self.conv2 = SAGEConv(
                in_channels=hidden_dim * heads, 
                out_channels=hidden_dim
            )
            self.dropout = Dropout(dropout)
            
        else:
            raise ValueError(f"Unsupported conv_type: {conv_type}. Choose from 'TransformerConv', 'GATConv', 'SAGEConv'")
        
        # Classifier head (same for all models)
        self.classifier = torch.nn.Sequential(
            Linear(hidden_dim, hidden_dim),
            torch.nn.ReLU(),
            Dropout(dropout),
            Linear(hidden_dim, num_classes)
        )
    
    def forward(self, x, edge_index, return_embedding=False):
        """
        Forward pass.
        
        Parameters:
        -----------
        x : torch.Tensor
            Node features
        edge_index : torch.Tensor
            Edge indices
        return_embedding : bool
            If True, return both logits and embeddings
        
        Returns:
        --------
        logits : torch.Tensor
            Classification logits
        embeddings : torch.Tensor (optional)
            Node embeddings (only if return_embedding=True)
        """
        # First conv layer
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        
        # SAGEConv needs manual dropout
        if self.conv_type == 'SAGEConv':
            x = self.dropout(x)
        
        # Second conv layer
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        
        # Classifier
        logits = self.classifier(x)
        
        if return_embedding:
            return logits, x  # logits, embeddings
        else:
            return logits

# Data preperation

In [ ]:
from torch_geometric.data import Data
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedShuffleSplit

# Features
if not isinstance(ad.X, np.ndarray):
    X = torch.tensor(ad.X.toarray(), dtype=torch.float32)
else:
    X = torch.tensor(ad.X, dtype=torch.float32)

# Graph edges
adj = ad.obsp["connectivities"].tocoo()
edge_index = torch.tensor(np.vstack((adj.row, adj.col)), dtype=torch.long)

# Labels
primary_mask = (ad.obs["Primary_or_Metastatic"] == "Primary").values
local_mask = (ad.obs["Primary_or_Metastatic"] == "Metastatic") & (ad.obs["source"] == "local").values
distant_mask = (ad.obs["Primary_or_Metastatic"] == "Metastatic") & (ad.obs["source"] == "distant").values

y_3class = np.full(ad.n_obs, -1, dtype=int)
y_3class[primary_mask] = 0
y_3class[local_mask] = 1
y_3class[distant_mask] = 2
y_3class = torch.tensor(y_3class, dtype=torch.long)

# Binary label for Primary vs Local mets only (for filtered evaluation)
y_binary = np.full(ad.n_obs, -1, dtype=int)
y_binary[primary_mask] = 0
y_binary[local_mask] = 1
y_binary = torch.tensor(y_binary, dtype=torch.long)
valid_mask = (y_3class == 0) | (y_3class == 1)
valid_idx = np.where(valid_mask)[0]

# ===== 10% Stratified Split (Primary vs Local only) =====
split = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_valid_idx, heldout_idx = next(split.split(valid_idx, y_3class[valid_idx]))

val_idx = valid_idx[heldout_idx]       # final val set = only 0/1 labels
train_idx = np.setdiff1d(np.arange(ad.n_obs), val_idx)  # everything else is training


data = Data(
    x=X,
    edge_index=edge_index,
    y=y_3class
)
data.train_idx = train_idx
data.val_idx = val_idx
data.y_binary = y_binary

print(f"Prepared {data.num_nodes} nodes")
print(f"Primary: {(y_3class==0).sum().item()} | Local: {(y_3class==1).sum().item()} | Distant: {(y_3class==2).sum().item()}")
print(len(train_idx), 'training nodes', len(val_idx), 'validation nodes')

# Training

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score


def evaluate(model, x_val, y_binary, device):
    """
    Evaluate model on PRIMARY (0) vs NON-PRIMARY/METS (1).
    All cells are included (Primary, Local, Distant).
    
    Parameters:
    -----------
    model : nn.Module
        Model to evaluate
    x_val : torch.Tensor
        Validation features
    y_binary : torch.Tensor
        Binary labels (0=Primary, 1=Mets including Local+Distant)
    device : torch.device
        Device
    
    Returns:
    --------
    acc, f1, auc, auprc : float
        Evaluation metrics
    y_true, y_pred : np.array
        True and predicted labels
    """
    model.eval()
    
    # Self-loop edges
    N = x_val.size(0)
    edge_index_val = torch.arange(N, device=device).unsqueeze(0).repeat(2, 1)
    
    with torch.no_grad():
        logits, _ = model(x_val, edge_index=edge_index_val, return_embedding=True)
    
    # Get probabilities and predictions
    y_prob = torch.softmax(logits, dim=1)
    y_pred = logits.argmax(dim=1)
    
    # Convert to numpy
    y_true = y_binary.cpu().numpy()
    y_pred = y_pred.cpu().numpy()
    
    # This is better for Primary vs (Local+Distant)
    y_prob_mets_combined = (y_prob[:, 1] + y_prob[:, 2]).cpu().numpy()
    
    # For binary evaluation, map predictions: 0→0, {1,2}→1
    y_pred_binary = (y_pred >= 1).astype(int)
    
    # Compute metrics
    acc = accuracy_score(y_true, y_pred_binary)
    f1 = f1_score(y_true, y_pred_binary, zero_division=0)
    
    # Check if we have both classes
    if len(np.unique(y_true)) < 2:
        auc = np.nan
        auprc = np.nan
    else:
        try:
            # Use combined probability (Local + Distant)
            auc = roc_auc_score(y_true, y_prob_mets_combined)
        except Exception as e:
            print(f"[ERROR] AUC: {e}")
            auc = np.nan
        
        try:
            auprc = average_precision_score(y_true, y_prob_mets_combined)
        except Exception as e:
            print(f"[ERROR] AUPRC: {e}")
            auprc = np.nan
    
    return acc, f1, auc, auprc, y_true, y_pred_binary

In [ ]:
import torch.optim as optim
from torch_geometric.loader import RandomNodeLoader
from torch_geometric.utils import subgraph
from tqdm import tqdm

def NT_Xent(embeddings, tau=0.5):
    device = embeddings.device
    z_i = F.normalize(embeddings, dim=1)
    z_j = F.normalize(embeddings[torch.randperm(z_i.size(0))], dim=1)

    logits = torch.mm(z_i, z_j.t()) / tau
    labels = torch.arange(z_i.size(0), device=device)
    loss = F.cross_entropy(logits, labels)
    return loss

def run_model(data, save_dir, epochs=100, seed=42, patience=10, gamma=0.1, hidden_dim = 256, tau = 0.05, conv_type='TransformerConv', dropout_rate = 0.3):
    os.makedirs(save_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    memory_usgae()

    train_idx = data.train_idx
    val_idx = data.val_idx

    train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    train_mask[train_idx] = True

    # Build train graph
    edge_index_train, _ = subgraph(train_mask, data.edge_index, relabel_nodes=True)
    x_train = data.x[train_mask]
    y_train = data.y[train_mask]

    train_data = Data(
        x=x_train,
        edge_index=edge_index_train,
        y=y_train
    )

    x_val = data.x[val_idx].to(device)
    y_val = data.y[val_idx].to(device)

    y_binary_val = data.y_binary[val_idx]


    model = scMeta(
        input_dim=data.num_node_features,
        hidden_dim=hidden_dim,
        num_classes=len(torch.unique(data.y)),
        conv_type=conv_type,
        dropout=dropout_rate
    ).to(device)

    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    best_acc = 0
    best_auc = 0.0
    best_auprc = 0.0
    patience_counter = 0
    best_model_path = os.path.join(save_dir, f"best_model.pt")

    # double check class distribution in val set
    # no distant mets should be here, no label 2
    y_val_counts = torch.bincount(data.y[val_idx])
    for label, count in enumerate(y_val_counts):
        print(f"Val label {label}: {count.item()} samples")

    # Pre-training eval
    acc, f1, auc, auprc, _, _ = evaluate(
        model, x_val, y_binary_val, device
    )
    print(f"Pre-training stats: Acc={acc:.4f}, F1={f1:.4f}, AUROC={auc:.4f}, AUPRC={auprc:.4f}")

    for epoch in tqdm(range(epochs)):
        model.train()
        total_loss = 0

        loader = RandomNodeLoader(train_data, num_parts=100, shuffle=True)
        for batch in loader:
            batch = batch.to(device)
            logits, emb = model(batch.x, batch.edge_index, return_embedding=True)

            loss_ce = F.cross_entropy(logits, batch.y)
            loss_con = NT_Xent(emb, tau=tau)
            loss = loss_ce + gamma * loss_con

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # Evaluate
        acc, f1, auc, auprc, _, _ = evaluate(
            model, x_val, y_binary_val, device
        )

        if epoch % 10 == 0:
            print(f"Epoch {epoch}: Acc={acc:.4f}, F1={f1:.4f}, AUROC={auc:.4f}, AUPRC={auprc:.4f}")

        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), best_model_path)
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"⏹️ Early stopping at epoch {epoch + 1}")
                break


In [ ]:
save_dir = '/scratch/gilbreth/wang3712/Metastasis_single_cell/scMeta_final_train_auc/'



run_model(
    data,
    save_dir=save_dir,
    epochs=200,
    patience=20,
    conv_type='TransformerConv'
)

# Visualize

In [ ]:
save_dir = '/scratch/gilbreth/wang3712/Metastasis_single_cell/scMeta_final_train_auc/'

best_model_path = save_dir+'/best_model.pt'
model = scMeta(
    input_dim=data.num_node_features,
    hidden_dim=256,
    num_classes=3,
    conv_type='TransformerConv'
).to(device)


# Load best model and evaluate on test set
model.load_state_dict(torch.load(best_model_path))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()


In [ ]:
model.eval()

full_loader = NeighborLoader(
    data,
    input_nodes=None,              # all nodes
    num_neighbors=[-1, -1],        # use full neighborhood
    batch_size=512,
    shuffle=False
)

all_embeddings = torch.zeros((data.num_nodes, model.classifier[0].in_features))

with torch.no_grad():
    for batch in tqdm(full_loader):
        batch = batch.to(device)
        node_ids = batch.n_id[:batch.batch_size]
        _, emb = model(batch.x, batch.edge_index, return_embedding=True)
        all_embeddings[node_ids.cpu()] = emb[:batch.batch_size].cpu()  # only keep target embeddings

all_embeddings.shape

In [ ]:
ad_emb = ann.AnnData(X=all_embeddings.numpy())
ad_emb.obs = ad.obs.copy()
# sc.pp.pca(ad_emb)
sc.pp.neighbors(ad_emb, use_rep='X')
sc.tl.umap(ad_emb)

In [ ]:
ad_emb.write_h5ad(save_dir+'/umap_embedding.h5ad', compression='gzip')

In [ ]:
ad_emb = sc.read_h5ad(save_dir+'/umap_embedding.h5ad')
ad_emb

In [ ]:
for obs in ['Final_cancer_type', 'Project_ID', 'Final_tissue', 'Classifier_label', 'source']:
    sc.pl.umap(ad_emb, color=obs)

In [ ]:
for obs in ['Final_cancer_type', 'Project_ID', 'Final_tissue', 'Classifier_label', 'source']:
    sc.pl.umap(ad_emb, color=obs)

In [ ]:
ad_emb.obs['Primary Cancer Type'] = ad_emb.obs['Final_cancer_type']
sc.set_figure_params(figsize=(3, 3))
sc.pl.umap(ad_emb, color='Primary Cancer Type', save='/Figure3.cancer_type.svg')

# Feature importance

## Identify Genes per cancer

In [ ]:
from torch_geometric.utils import subgraph
import torch
import pandas as pd
import numpy as np

def compute_gene_importance_subgraph_cpu(model, data, gene_names, mask, return_classwise=True):
    """
    Compute per-gene gradient attribution for a subgraph defined by `mask`.
    Also returns cell-by-gene gradient matrix for visualization.
    """
    model.eval()
    model_cpu = model.to("cpu")  # force model to CPU

    # 1. Get node indices from mask
    indices = mask.nonzero(as_tuple=True)[0]

    # 2. Extract subgraph and relabel
    edge_index_sub, _ = subgraph(indices, data.edge_index, relabel_nodes=True)
    edge_index_sub = edge_index_sub.to("cpu")

    # 3. Subset features and labels
    x_sub = data.x[indices].to("cpu")
    labels_sub = data.y_binary[indices].to("cpu")

    # 4. Require gradients
    x_sub = x_sub.clone().detach().requires_grad_(True)

    # 5. Forward and backward pass
    logits = model_cpu(x_sub, edge_index_sub)
    loss = torch.nn.functional.cross_entropy(logits, labels_sub)
    loss.backward()

    grads = x_sub.grad.detach().numpy()  # [n_cells, n_genes]
    labels_np = labels_sub.detach().numpy()

    # Create cell-by-gene gradient matrix
    grad_df = pd.DataFrame(np.abs(grads), columns=gene_names)
    grad_df["Label"] = labels_np  # add label for grouping if needed

    # Compute gene-level summary if requested
    if return_classwise:
        primary_mask = (labels_np == 0)
        mets_mask = (labels_np == 1)

        primary_grad = np.mean(np.abs(grads[primary_mask]), axis=0)
        mets_grad = np.mean(np.abs(grads[mets_mask]), axis=0)
        delta_grad = mets_grad - primary_grad

        ranking = pd.DataFrame({
            "Gene": gene_names,
            "PrimaryGrad": primary_grad,
            "MetsGrad": mets_grad,
            "DeltaGrad": delta_grad
        }).sort_values("DeltaGrad", ascending=False)

        return ranking, grad_df
    else:
        importance = np.mean(np.abs(grads), axis=0)
        ranking = pd.DataFrame({
            "Gene": gene_names,
            "Importance": importance
        }).sort_values("Importance", ascending=False)

        return ranking, grad_df


In [ ]:
save_dir = '/scratch/gilbreth/wang3712/Metastasis_single_cell/scMeta_final_train_auc/'
best_hidden_dim= 256
best_model_path = save_dir+'/best_model.pt'
model = scMeta(
    input_dim=data.num_node_features,
    hidden_dim=best_hidden_dim,
    num_classes=3
).to(device)


# Load best model and evaluate on test set
model.load_state_dict(torch.load(best_model_path))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()


In [ ]:
ranking_per_cancer = {}
grad_per_cancer = {}
for cancer in np.unique(ad.obs["Final_cancer_type"]):
    memory_usgae()
    cancer_mask = (ad.obs["Final_cancer_type"] == cancer) & (data.y_binary.cpu().numpy() != -1)
    cancer_mask = torch.tensor(cancer_mask.values, dtype=torch.bool)

    if cancer_mask.sum() < 30:
        continue

    print(f"Computing gene gradients for {cancer} ({cancer_mask.sum().item()} cells)")
    
    ranking, grad_df = compute_gene_importance_subgraph_cpu(
        model=model,
        data=data,
        gene_names=ad.var_names.tolist(),
        mask=cancer_mask,
        return_classwise=True
    )


    ranking_per_cancer[cancer] = ranking
    grad_per_cancer[cancer] = grad_df

In [ ]:
memory_usgae()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import os

output_dir = "delta_grad_plots"
os.makedirs(output_dir, exist_ok=True)

for cancer in ranking_per_cancer:
    df = ranking_per_cancer[cancer]
    df_sorted = df.sort_values("DeltaGrad", ascending=True)

    x = np.arange(len(df_sorted))
    y = df_sorted["DeltaGrad"].values
    print(cancer)
    print(df_sorted.tail(n=5))

    size_scale = 5e8
    sizes = np.clip(y * size_scale, 5, 150)

    fig, ax = plt.subplots(figsize=(2, 2))  # square figure size
    # Removed ax.set_aspect('equal') to avoid cutoff issues

    ax.scatter(x, y, s=sizes, color="dodgerblue", alpha=0.5,
               edgecolors="black", linewidths=0.2)
    
    # Pad the top of the y-axis to avoid clipping
    y_max = y.max()
    ax.set_ylim(0, y_max * 1.1)  # 10% headroom
    ax.set_title(cancer)
    ax.set_xlabel("Gene Rank")
    ax.set_ylabel("Δ Gradient")
    ax.grid(False)

    # Scientific notation formatting
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
    ax.yaxis.offsetText.set_visible(False)
    formatter = ticker.ScalarFormatter(useMathText=True)
    formatter.set_scientific(True)
    formatter.set_powerlimits((0, 0))
    ax.yaxis.set_major_formatter(formatter)

    # Manual offset label
    y_offset = ax.yaxis.get_offset_text().get_text()
    if y_offset:
        ax.annotate(y_offset, xy=(0, 1.02), xycoords='axes fraction',
                    ha='left', fontsize=10)

    # Layout and save
    # plt.tight_layout()
    filename = f"{cancer.replace(' ', '_')}_delta_grad.png"
    # plt.show()
    plt.savefig(os.path.join(output_dir, filename), dpi=600, bbox_inches='tight')
    plt.close()


In [ ]:
# find top genes
top_n = 200
top_genes_dict = dict()

for cancer in ranking_per_cancer.keys():
    print('Processing ',cancer)
    top_genes =ranking_per_cancer[cancer].head(top_n)["Gene"].to_list()
    # print(' '.join(top_genes))
    top_genes_dict[cancer] = top_genes
top_genes_dict = pd.DataFrame(top_genes_dict)
top_genes_dict

In [ ]:
top_genes_dict.head(50).to_csv('top_genes.csv')

In [ ]:
all_top_genes = set(top_genes_dict.values.ravel())
len(all_top_genes)

In [ ]:
shared_genes = set(top_genes_dict['Breast Cancer']) & set(top_genes_dict['Colorectal Cancer']) & set(top_genes_dict['Lung Cancer']) & set(top_genes_dict['Ovarian Cancer'])

print("Shared genes across all four cancer types:")
print(shared_genes)

In [ ]:
# Filter samples where metastasis is either 'Primary' or 'local'
mask = ad.obs["source"] != "distant"
adata_sub = ad[mask].copy()
# Ensure all gene names are strings and in the var_names index
filtered_genes = [gene for gene in shared_genes if gene in adata_sub.var_names]

# Subset the genes
adata_sub = adata_sub[:, filtered_genes].copy()
adata_sub

In [ ]:
adata_sub.obs['extra_label'] = adata_sub.obs['Final_tissue'].astype(str) + ' ' + adata_sub.obs['source'].astype(str)

## Pan-cancer

In [ ]:
top_n = 1000

top_genes_per_cancer = {
    cancer: set(df.head(top_n)["Gene"]) for cancer, df in ranking_per_cancer.items()
}
shared_genes = set.intersection(*top_genes_per_cancer.values())
print(f"Shared genes across all cancers (top {top_n}): {len(shared_genes)}")


In [ ]:
shared_genes = list(shared_genes)
shared_genes.sort()

In [ ]:
import json
import requests
import time

In [ ]:
genes_str = '\n'.join(shared_genes)
# print(new_genes[:5])
description = 'pancancer_sigs'
payload = {
    'list': (None, genes_str),
    'description': (None, description)
}
ENRICHR_URL = 'https://maayanlab.cloud/Enrichr/addList'


response = requests.post(ENRICHR_URL, files=payload)
if not response.ok:
    raise Exception('Error analyzing gene list')

data = json.loads(response.text)

user_list_id = data['userListId']   
# print(user_list_id)
ENRICHR_URL = 'https://maayanlab.cloud/Enrichr/view?userListId=%s'

response = requests.get(ENRICHR_URL % user_list_id)
if not response.ok:
    raise Exception('Error getting gene list')

data = json.loads(response.text)
# user_list_id = data['userListId']   


ENRICHR_URL = 'https://maayanlab.cloud/Enrichr/enrich'
query_string = '?userListId=%s&backgroundType=%s'
gene_set_library = 'GO_Biological_Process_2025'
response = requests.get(
    ENRICHR_URL + query_string % (user_list_id, gene_set_library)
 )

if not response.ok:
    raise Exception('Error fetching enrichment results')

# Rank, Term name, P-value, Odds ratio, Combined score, 
# Overlapping genes, Adjusted p-value, Old p-value, Old adjusted p-value
data = json.loads(response.text)
   

In [ ]:
pathway_df = pd.DataFrame.from_dict(data[gene_set_library])
colomns = ['Rank', 'Term name', 'P-value', 'Odds ratio', 'Combined score', 'Overlapping genes', 'Adjusted p-value', 
           'Old p-value','Old adjusted p-value']
pathway_df.columns = colomns
pathway_df = pathway_df.set_index('Rank')
pathway_df

## Per-cancer

In [ ]:
import json
import requests
import time

In [ ]:
top_n = 500
all_results = dict()

for cancer in ranking_per_cancer.keys():
    print('Processing ',cancer)
    top_genes =ranking_per_cancer[cancer].head(top_n)["Gene"]
    genes_str = '\n'.join(top_genes)
    # print(new_genes[:5])
    description = cancer+'_top'+str(top_n)
    payload = {
        'list': (None, genes_str),
        'description': (None, description)
    }
    ENRICHR_URL = 'https://maayanlab.cloud/Enrichr/addList'


    response = requests.post(ENRICHR_URL, files=payload)
    if not response.ok:
        raise Exception('Error analyzing gene list')

    data = json.loads(response.text)

    user_list_id = data['userListId']   
    # print(user_list_id)
    ENRICHR_URL = 'https://maayanlab.cloud/Enrichr/view?userListId=%s'

    response = requests.get(ENRICHR_URL % user_list_id)
    if not response.ok:
        raise Exception('Error getting gene list')

    data = json.loads(response.text)
    # user_list_id = data['userListId']   


    ENRICHR_URL = 'https://maayanlab.cloud/Enrichr/enrich'
    query_string = '?userListId=%s&backgroundType=%s'
    gene_set_library = 'GO_Biological_Process_2025'
    response = requests.get(
        ENRICHR_URL + query_string % (user_list_id, gene_set_library)
     )

    if not response.ok:
        raise Exception('Error fetching enrichment results')

    # Rank, Term name, P-value, Odds ratio, Combined score, 
    # Overlapping genes, Adjusted p-value, Old p-value, Old adjusted p-value
    data = json.loads(response.text)
    pathway_df = pd.DataFrame.from_dict(data[gene_set_library])
    colomns = ['Rank', 'Term name', 'P-value', 'Odds ratio', 'Combined score', 'Overlapping genes', 'Adjusted p-value', 
               'Old p-value','Old adjusted p-value']
    pathway_df.columns = colomns
    pathway_df = pathway_df.set_index('Rank')
    all_results[cancer] = pathway_df
   

In [ ]:
# Quick version
pathway_data = {}
for cancer, df in all_results.items():
    for _, row in df[df['Adjusted p-value'] < 0.05].iterrows():
        pathway = row['Term name']
        if pathway not in pathway_data:
            pathway_data[pathway] = {}
        pathway_data[pathway][cancer] = -np.log10(row['Adjusted p-value'])

df_scmeta = pd.DataFrame([
    {'Pathway': p, **{c: vals.get(c, 0) for c in all_results.keys()}, 'N_Cancers': len(vals)}
    for p, vals in pathway_data.items() if len(vals) >= 2
]).sort_values('N_Cancers', ascending=False)

df_scmeta.to_csv('scMeta_pathways.csv', index=False)
df_scmeta.head(20)


In [ ]:
from collections import defaultdict

# Step 1: collect pathways per cancer
pathway_to_cancers = defaultdict(set)
adj_pval_cutoff = 0.05

for cancer, df in all_results.items():
    df['Adjusted p-value'] = pd.to_numeric(df['Adjusted p-value'], errors='coerce')
    sig_df = df[df['Adjusted p-value'] < adj_pval_cutoff]

    for term in sig_df['Term name']:
        pathway_to_cancers[term].add(cancer)

# Step 2: find pathways enriched in >= 3 cancers
shared_pathways = {pathway: cancers for pathway, cancers in pathway_to_cancers.items() if len(cancers) > 1}

# Step 3: optionally convert to DataFrame for display
import pandas as pd
df_shared = pd.DataFrame([
    {"Pathway": k, "Num_Cancers": len(v), "Cancers": ", ".join(sorted(v))}
    for k, v in shared_pathways.items()
]).sort_values("Num_Cancers", ascending=False)

df_shared

In [ ]:
import pandas as pd
import os

os.makedirs("cytoscape_files", exist_ok=True)

for cancer, df in all_results.items():
    out_path = f"cytoscape_files/{cancer}_enrichment.txt"
    
    # Create required columns
    df_out = pd.DataFrame()
    df_out["GO.ID"] = df["Term name"]
    df_out["GS_DSCR"] = df["Term name"].fillna("").apply(lambda x: x.split(' (')[0])
    # df_out["NES"] = df["Combined score"]  # or use a proxy like enrichment score
    df_out["p.Val"] = df["P-value"]
    df_out["FDR"] = df["Adjusted p-value"]
    df_out = df_out[df_out["GO.ID"].isin(pathways)]

    # df_out["HITS"] = df["Overlapping genes"]  # comma-separated list
    
    df_out.to_csv(out_path, sep="\t", index=False)
    print(f"Saved: {out_path}")


# Tranditional DEG Analysis

In [ ]:
ad

In [ ]:
import json
import requests
import time

In [ ]:
degs_per_cancer = {}

for cancer in ad.obs['Final_cancer_type'].unique():
    # Step 1: Subset to this cancer type
    ad_sub = ad[ad.obs['Final_cancer_type'] == cancer].copy()

    # Step 2: Keep only source == Primary or local
    is_primary_or_local = ad_sub.obs['source'].isin(['Primary', 'local'])
    ad_filtered = ad_sub[is_primary_or_local].copy()

    # Skip if not enough cells
    if ad_filtered.n_obs < 30:
        print(f"Skipping {cancer} due to low cell count ({ad_filtered.n_obs})")
        continue

    # Step 3: Set group labels based on source
    ad_filtered.obs['group'] = ad_filtered.obs['source'].astype('category')
    ad_filtered.obs['group'] = ad_filtered.obs['group'].cat.set_categories(['Primary', 'local'])

    print(f"Running DEGs for {cancer} (n={ad_filtered.n_obs})")
    ad_filtered.uns['log1p'] = {'base': np.e}

    # Step 4: Run DEG analysis
    sc.tl.rank_genes_groups(
        ad_filtered,
        groupby='group',
        method='wilcoxon',
        reference='Primary',
        pts=True,
        use_raw = False
    )

    # Step 5: Save results
    result_df = sc.get.rank_genes_groups_df(ad_filtered, group='local')
    degs_per_cancer[cancer] = result_df


In [ ]:
top_n = 500
all_results_deg = dict()

for cancer in degs_per_cancer.keys():
    print('Processing ',cancer)
    # Top N genes by absolute scores
    top_genes = (
        degs_per_cancer[cancer]
        .assign(abs_score=lambda df: df['scores'].abs())
        .sort_values('abs_score', ascending=False)
        .head(top_n)['names']
        .tolist()
    )

    genes_str = '\n'.join(top_genes)
    # print(new_genes[:5])
    description = cancer+'_top'+str(top_n)
    payload = {
        'list': (None, genes_str),
        'description': (None, description)
    }
    ENRICHR_URL = 'https://maayanlab.cloud/Enrichr/addList'


    response = requests.post(ENRICHR_URL, files=payload)
    if not response.ok:
        raise Exception('Error analyzing gene list')

    data = json.loads(response.text)

    user_list_id = data['userListId']   
    # print(user_list_id)
    ENRICHR_URL = 'https://maayanlab.cloud/Enrichr/view?userListId=%s'

    response = requests.get(ENRICHR_URL % user_list_id)
    if not response.ok:
        raise Exception('Error getting gene list')

    data = json.loads(response.text)
    # user_list_id = data['userListId']   


    ENRICHR_URL = 'https://maayanlab.cloud/Enrichr/enrich'
    query_string = '?userListId=%s&backgroundType=%s'
    gene_set_library = 'GO_Biological_Process_2025'
    response = requests.get(
        ENRICHR_URL + query_string % (user_list_id, gene_set_library)
     )

    if not response.ok:
        raise Exception('Error fetching enrichment results')

    # Rank, Term name, P-value, Odds ratio, Combined score, 
    # Overlapping genes, Adjusted p-value, Old p-value, Old adjusted p-value
    data = json.loads(response.text)
    pathway_df = pd.DataFrame.from_dict(data[gene_set_library])
    colomns = ['Rank', 'Term name', 'P-value', 'Odds ratio', 'Combined score', 'Overlapping genes', 'Adjusted p-value', 
               'Old p-value','Old adjusted p-value']
    pathway_df.columns = colomns
    pathway_df = pathway_df.set_index('Rank')
    all_results_deg[cancer] = pathway_df
   

In [ ]:
from collections import defaultdict

# Step 1: collect pathways per cancer
pathway_to_cancers = defaultdict(set)
adj_pval_cutoff = 0.05

for cancer, df in all_results_deg.items():
    df['Adjusted p-value'] = pd.to_numeric(df['Adjusted p-value'], errors='coerce')
    sig_df = df[df['Adjusted p-value'] < adj_pval_cutoff]

    for term in sig_df['Term name']:
        pathway_to_cancers[term].add(cancer)

# Step 2: find pathways enriched in >= 3 cancers
shared_pathways = {pathway: cancers for pathway, cancers in pathway_to_cancers.items() if len(cancers) > 1}

# Step 3: optionally convert to DataFrame for display
import pandas as pd
df_shared = pd.DataFrame([
    {"Pathway": k, "Num_Cancers": len(v), "Cancers": ", ".join(sorted(v))}
    for k, v in shared_pathways.items()
]).sort_values("Num_Cancers", ascending=False)

df_shared

In [ ]:
pathways = set(df_shared['Pathway'].values)
len(pathways)

In [ ]:
import pandas as pd
import os

os.makedirs("cytoscape_files_deg/", exist_ok=True)

for cancer, df in all_results_deg.items():
    out_path = f"cytoscape_files_deg/{cancer}_enrichment.txt"
    
    # Create required columns
    df_out = pd.DataFrame()
    df_out["GO.ID"] = df["Term name"]
    df_out["GS_DSCR"] = df["Term name"].fillna("").apply(lambda x: x.split(' (')[0])
    # df_out["NES"] = df["Combined score"]  # or use a proxy like enrichment score
    df_out["p.Val"] = df["P-value"]
    df_out["FDR"] = df["Adjusted p-value"]
    df_out = df_out[df_out["GO.ID"].isin(pathways)]
    # df_out["HITS"] = df["Overlapping genes"]  # comma-separated list
    
    df_out.to_csv(out_path, sep="\t", index=False)
    print(f"Saved: {out_path}")


In [ ]:
# Quick version
pathway_data = {}
for cancer, df in all_results_deg.items():
    for _, row in df[df['Adjusted p-value'] < 0.05].iterrows():
        pathway = row['Term name']
        if pathway not in pathway_data:
            pathway_data[pathway] = {}
        pathway_data[pathway][cancer] = -np.log10(row['Adjusted p-value'])

df_scmeta = pd.DataFrame([
    {'Pathway': p, **{c: vals.get(c, 0) for c in all_results.keys()}, 'N_Cancers': len(vals)}
    for p, vals in pathway_data.items() if len(vals) >= 2
]).sort_values('N_Cancers', ascending=False)

df_scmeta.to_csv('scMeta_pathways.deg.csv', index=False)
df_scmeta.head(20)
